# A tour of the build

Read-only: nothing depends on this notebook. It replays the `benchmark_data` build and shows the tables and the consumer views. The real entry point is `python -m benchmark_data build`.

**Kernel.** Use the repository's virtual environment: `uv venv --python 3.11 .venv && uv pip install -e . --group dev`, then pick the `benchmark-data-pipeline (.venv)` kernel (registered with `python -m ipykernel install --user --name benchmark-data-pipeline`). The first cell also puts `1_pipeline/` on the path, so any interpreter that has pandas works.

In [ ]:
import logging
import sys
from pathlib import Path

# The package lives in 1_pipeline/; make it importable even without `pip install -e .`.
REPO = Path.cwd().resolve()
while not (REPO / "1_pipeline").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "1_pipeline"))

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout)

from benchmark_data import config  # noqa: E402
from benchmark_data.build import build  # noqa: E402

build()

In [ ]:
import pandas  # noqa: E402

tables = {f: pandas.read_csv(config.DATABASE_DIR / f) for f in config.TABLE_FILES}
print({f: len(t) for f, t in tables.items()})
tables["scores.csv"].head()

## A name, translated in front of you

The source's `aliases.csv` first, otherwise the machine-name grammar (`identity.split_name`). `''` = unknown.

In [ ]:
from benchmark_data.identity import load_aliases, resolve_model  # noqa: E402

examples = [("epoch", "claude-opus-4-8_high"), ("epoch", "zai-org/glm-4.7"),
            ("epoch", "claude-opus-4-5-20251101_unknown"), ("kaggle", "claude-opus-4-8"),
            ("seal", "Opus 4.8"), ("rand", "Claude 3.7 Sonnet V1 - Reasoning 16000"),
            ("epoch_cyber", "GPT-5.5-Cyber")]
rows = []
for source, name in examples:
    aliases = load_aliases(config.FEED_DIR / source / "aliases.csv")
    status, ident, harness = resolve_model(name, aliases, config.SPLITTABLE[source])
    rows.append({"source": source, "raw name": name, "status": status, "harness": harness,
                 **(ident or {})})
pandas.DataFrame(rows)

## The consumer views

`3_views/all_scores_flat.csv` (every score, Multiaxis_ECI's columns plus `source_url`, `feed`, `lower_bound`, `ceiling`) and `3_views/dated_scores_flat.csv` (dated models only, benchmark-forecasting's input).

In [ ]:
views = {f: pandas.read_csv(config.VIEWS_DIR / f) for f in config.VIEW_FILES}
print({f: len(v) for f, v in views.items()})
views["dated_scores_flat.csv"].sample(8, random_state=1)